# Load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(".")

train = pd.read_csv(DATA_DIR / "train_with_player_bio.csv")

train.head()

,SEASON_1,SEASON_2,TEAM_ID,TEAM_NAME,PLAYER_ID,PLAYER_NAME,POSITION_GROUP,POSITION,GAME_DATE,GAME_ID,...,IS_THREE,PLAYER_HEIGHT,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,AGE,COUNTRY,COLLEGE,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER
0,2022,2021-22,1610612737,Atlanta Hawks,1628381,John Collins,F,F-C,10-21-2021,22100014,...,0,6-9,81.0,226.0,24.0,USA,Wake Forest,2017.0,1.0,19.0
1,2022,2021-22,1610612737,Atlanta Hawks,1629631,De'Andre Hunter,F,F-G,10-21-2021,22100014,...,0,6-8,80.0,221.0,24.0,USA,Virginia,2019.0,1.0,4.0
2,2022,2021-22,1610612737,Atlanta Hawks,1629027,Trae Young,G,G,10-21-2021,22100014,...,0,6-1,73.0,164.0,23.0,USA,Oklahoma,2018.0,1.0,5.0
3,2022,2021-22,1610612737,Atlanta Hawks,1628381,John Collins,F,F-C,10-21-2021,22100014,...,0,6-9,81.0,226.0,24.0,USA,Wake Forest,2017.0,1.0,19.0
4,2022,2021-22,1610612737,Atlanta Hawks,203992,Bogdan Bogdanović,G,G,10-21-2021,22100014,...,1,6-6,78.0,225.0,29.0,Serbia,NaN,2014.0,1.0,27.0


In [2]:
# Inspect columns and coordinate scales 
print('Train:', train.shape)
print('All columns:', train.columns.tolist())

Train: (872169, 40)
All columns: ['SEASON_1', 'SEASON_2', 'TEAM_ID', 'TEAM_NAME', 'PLAYER_ID', 'PLAYER_NAME', 'POSITION_GROUP', 'POSITION', 'GAME_DATE', 'GAME_ID', 'GAME_EVENT_ID', 'HOME_TEAM', 'AWAY_TEAM', 'EVENT_TYPE', 'SHOT_MADE', 'ACTION_TYPE', 'SHOT_TYPE', 'BASIC_ZONE', 'ZONE_NAME', 'ZONE_ABB', 'ZONE_RANGE', 'LOC_X', 'LOC_Y', 'SHOT_DISTANCE', 'QUARTER', 'MINS_LEFT', 'SECS_LEFT', 'SEASON', 'MADE', 'SECONDS_LEFT_IN_PERIOD', 'IS_THREE', 'PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'AGE', 'COUNTRY', 'COLLEGE', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER']


# Location Feature Engineering

The data file contain 40 columns(features). 

We will add exact distance, shot angle, corner-three flag, and one-hot shot zones. 

In [3]:
print(train[["LOC_X", "LOC_Y", "SHOT_DISTANCE", "IS_THREE", "ZONE_NAME"]].head(10))

   LOC_X  LOC_Y  SHOT_DISTANCE  IS_THREE          ZONE_NAME
0   -0.4  17.25             12         0             Center
1   -5.0  26.45             21         0             Center
2   -3.7   6.75              3         0             Center
3   14.5  13.65             16         0          Left Side
4  -22.8  11.65             23         1         Right Side
5   -9.1  28.45             24         1  Right Side Center
6    0.0  13.25              8         0             Center
7    0.0   5.25              0         0             Center
8   -4.5  29.85             25         1             Center
9    3.7  17.35             12         0             Center


In [4]:
print(train[[
    "LOC_X", "LOC_Y", "SHOT_DISTANCE",
    "ZONE_NAME", "IS_THREE"
]].head(10))

   LOC_X  LOC_Y  SHOT_DISTANCE          ZONE_NAME  IS_THREE
0   -0.4  17.25             12             Center         0
1   -5.0  26.45             21             Center         0
2   -3.7   6.75              3             Center         0
3   14.5  13.65             16          Left Side         0
4  -22.8  11.65             23         Right Side         1
5   -9.1  28.45             24  Right Side Center         1
6    0.0  13.25              8             Center         0
7    0.0   5.25              0             Center         0
8   -4.5  29.85             25             Center         1
9    3.7  17.35             12             Center         0


### 1. Exact Distance

`LOC_X`, `LOC_Y` are in feet. The goal is at (05.25)

In [5]:
train["exact_distance"] = np.sqrt(train["LOC_X"]**2 + (train["LOC_Y"] - 5.25)**2)

print(train[["LOC_X", "LOC_Y","SHOT_DISTANCE", "exact_distance"]].head())

   LOC_X  LOC_Y  SHOT_DISTANCE  exact_distance
0   -0.4  17.25             12       12.006665
1   -5.0  26.45             21       21.781644
2   -3.7   6.75              3        3.992493
3   14.5  13.65             16       16.757386
4  -22.8  11.65             23       23.681216


They are little different between `SHOT_DISTANCE` and `exact_distance`

### 2. Shot Angle

0 is straight ahead and 90 is from the side. 

In [6]:
train["shot_angle"] = np.degrees(np.arctan2(abs(train["LOC_X"]), abs(train["LOC_Y"] - 5.25)))

print(train[["LOC_X", "LOC_Y", "shot_angle"]].head())

   LOC_X  LOC_Y  shot_angle
0   -0.4  17.25    1.909152
1   -5.0  26.45   13.270644
2   -3.7   6.75   67.932100
3   14.5  13.65   59.915843
4  -22.8  11.65   74.320476


### 3. Corner Three Flag

This is an rule: `IS_THREE`is 1, at least 22 ft sideways, and near the baseline(`LOC_Y` <= 14)

In [7]:
train["is_corner_three"] = ((train["IS_THREE"] == 1) & (abs(train["LOC_X"]) >= 22) & (train["LOC_Y"] <= 14)).astype(int)

print(train["is_corner_three"].value_counts())

is_corner_three
0    783960
1     88209
Name: count, dtype: int64


### **4. Split training set into training and validation set.**

In [8]:
training = train[train["SEASON_1"].isin([2022, 2023, 2024])].copy()

validation = train[train["SEASON_1"] == 2025].copy()

print("Training:", training.shape)
print("Validation:", validation.shape)

Training: (652642, 43)
Validation: (219527, 43)


### 5. Shot Zone One-Hot Encoding

Convert the categorical shot zone features (`BASIC_ZONE`, `ZONE_RANGE`, and `ZONE_NAME`) into binary dummy columns using one-hot encoding.

In [9]:
zone_cols = ["BASIC_ZONE", "ZONE_RANGE", "ZONE_NAME"]

train_zones = pd.get_dummies(training[zone_cols], dtype=int)
val_zones = pd.get_dummies(validation[zone_cols], dtype=int)

val_zones = val_zones.reindex(columns=train_zones.columns, fill_value=0)

training = pd.concat([training, train_zones], axis=1)
validation = pd.concat([validation, val_zones], axis=1)

print("New zone features:", len(train_zones.columns))
print(train_zones.head())

New zone features: 18
   BASIC_ZONE_Above the Break 3  BASIC_ZONE_Backcourt  \
0                             0                     0   
1                             0                     0   
2                             0                     0   
3                             0                     0   
4                             0                     0   

   BASIC_ZONE_In The Paint (Non-RA)  BASIC_ZONE_Left Corner 3  \
0                                 1                         0   
1                                 0                         0   
2                                 0                         0   
3                                 0                         0   
4                                 0                         0   

   BASIC_ZONE_Mid-Range  BASIC_ZONE_Restricted Area  \
0                     0                           0   
1                     1                           0   
2                     0                           1   
3                     1 

In [10]:
print("Training:", training.shape)
print("Validation:", validation.shape)
print("Zone features:", len(train_zones.columns))

assert training.columns.equals(validation.columns)

print("All columns match!")

Training: (652642, 61)
Validation: (219527, 61)
Zone features: 18
All columns match!


### 6. Save

Keep the original CSV unchanged. make new output file including the original columns and new location features

In [11]:
print(train[["exact_distance", "shot_angle", "is_corner_three"]].head())
print("Training shape:", training.shape)
print("Validation shape:", validation.shape)

training.to_csv(
    "training_with_location_features.csv", index=False
)

validation.to_csv(
    "validation_with_location_features.csv", index=False
)

print("Saved!")

   exact_distance  shot_angle  is_corner_three
0       12.006665    1.909152                0
1       21.781644   13.270644                0
2        3.992493   67.932100                0
3       16.757386   59.915843                0
4       23.681216   74.320476                1
Training shape: (652642, 61)
Validation shape: (219527, 61)
Saved!


---

# Create some Models and Comparison

In [12]:
print(train[["SHOT_DISTANCE", "exact_distance", "shot_angle", "is_corner_three"]].head(10))

   SHOT_DISTANCE  exact_distance  shot_angle  is_corner_three
0             12       12.006665    1.909152                0
1             21       21.781644   13.270644                0
2              3        3.992493   67.932100                0
3             16       16.757386   59.915843                0
4             23       23.681216   74.320476                1
5             24       24.920875   21.417166                0
6              8        8.000000    0.000000                0
7              0        0.000000    0.000000                0
8             25       25.008199   10.366323                0
9             12       12.653063   17.002853                0


# Logistic Regression for accuracy, AUC

### Create some cases
1. Distance
2. Distance + shot_angle
3. Distance + shot_angle + corner_three
4. Distance + shot_angle + corner_three + zone

In [13]:
features1 = ["exact_distance"]

features2 = ["exact_distance", "shot_angle"]

features3 = ["exact_distance", "shot_angle", "is_corner_three"]

train_zones: One-hot encoding

In [14]:
zone_features = list(train_zones.columns)
zone_features

features4 = features3 + zone_features

### Compute Logistic Regression

In [15]:
%pip install scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Distance
model = LogisticRegression(max_iter=1000)

X_train = training[features1]
y_train = training["MADE"]

X_val = validation[features1]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

Accuracy: 0.6023450418399559
AUC: 0.6422602980361035


In [17]:
# 2. Distance + shot_angle
model = LogisticRegression(max_iter=1000)

X_train = training[features2]
y_train = training["MADE"]

X_val = validation[features2]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

Accuracy: 0.6021901633967576
AUC: 0.6421475552902338


In [18]:
# 3. Distance + shot_angle + corner_three
model = LogisticRegression(max_iter=1000)

X_train = training[features3]
y_train = training["MADE"]

X_val = validation[features3]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

Accuracy: 0.602723127451293
AUC: 0.6408298407966615


In [19]:
# 4. Distance + shot_angle + corner_three + zone
model = LogisticRegression(max_iter=1000)

X_train = training[features4]
y_train = training["MADE"]

X_val = validation[features4]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

Accuracy: 0.6229256537920165
AUC: 0.640821500170703


### Compute probabilities for Log Loss and Brier Score

In [20]:
from sklearn.metrics import log_loss, brier_score_loss

In [21]:
# 1. Distance
model = LogisticRegression(max_iter=1000)

X_train = training[features1]
y_train = training["MADE"]

X_val = validation[features1]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

print("Log Loss:", log_loss(y_val, probabilities))
print("Brier Score:", brier_score_loss(y_val, probabilities))

Accuracy: 0.6023450418399559
AUC: 0.6422602980361035
Log Loss: 0.6632129229589413
Brier Score: 0.23534399252187235


In [22]:
# 2. Distance + shot_angle
model = LogisticRegression(max_iter=1000)

X_train = training[features2]
y_train = training["MADE"]

X_val = validation[features2]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

print("Log Loss:", log_loss(y_val, probabilities))
print("Brier Score:", brier_score_loss(y_val, probabilities))

Accuracy: 0.6021901633967576
AUC: 0.6421475552902338
Log Loss: 0.6632244460904211
Brier Score: 0.2353480462387506


In [23]:
# 3. Distance + shot_angle + corner_three
model = LogisticRegression(max_iter=1000)

X_train = training[features3]
y_train = training["MADE"]

X_val = validation[features3]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

print("Log Loss:", log_loss(y_val, probabilities))
print("Brier Score:", brier_score_loss(y_val, probabilities))

Accuracy: 0.602723127451293
AUC: 0.6408298407966615
Log Loss: 0.6630408373838573
Brier Score: 0.23525627753234032


In [24]:
# 4. Distance + shot_angle + corner_three + Short zone
model = LogisticRegression(max_iter=1000)

X_train = training[features4]
y_train = training["MADE"]

X_val = validation[features4]
y_val = validation["MADE"]

model.fit(X_train, y_train)

predictions = model.predict(X_val)
probabilities = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, predictions))
print("AUC:", roc_auc_score(y_val, probabilities))

print("Log Loss:", log_loss(y_val, probabilities))
print("Brier Score:", brier_score_loss(y_val, probabilities))

Accuracy: 0.6229256537920165
AUC: 0.640821500170703
Log Loss: 0.6558818265566073
Brier Score: 0.23187967185943137


Extra case: Distance + zone

In [25]:
# 5. Distance + Shot Zones
features5 = ["exact_distance"] + zone_features

model = LogisticRegression(max_iter=1000)

model.fit(training[features5], training["MADE"])

predictions = model.predict(validation[features5])
probabilities = model.predict_proba(validation[features5])[:, 1]

print("Accuracy:", accuracy_score(validation["MADE"], predictions))
print("AUC:", roc_auc_score(validation["MADE"], probabilities))
print("Log Loss:", log_loss(validation["MADE"], probabilities))
print("Brier Score:", brier_score_loss(validation["MADE"], probabilities))

Accuracy: 0.622743443858842
AUC: 0.6411179321658708
Log Loss: 0.6559528875543683
Brier Score: 0.23191924712894324


# Conclusion


| Experiment | Accuracy ↑ | AUC ↑ | Log Loss ↓ | Brier ↓ |
|---|---:|---:|---:|---:|
| 1. Distance | 60.235% | **0.64226** | 0.66321 | 0.23534 |
| 2. Distance + shot_angle | 60.219% | 0.64215 | 0.66322 | 0.23535 |
| 3. Distance + shot_angle + corner_three | 60.272% | 0.64083 | 0.66304 | 0.23526 |
| **4. Distance + shot_angle + corner_three + Shot zone** | 62.293% | 0.64082 | **0.65588** | **0.23188** |
|  5. Distance + Shot Zones | 62.274% | 0.64112 | 0.65595 | 0.23192 |


`Shot zone` feature provivded the largest impact in model performance, increasing accuracy from 60.235% to 62.293%, while reducing Log Loss and Brier.

`shot_angle` and `corner_three` features provided the limited improvement. 

The `Distance` + `Shot Zones` model achieved the almost same performance as the full location feature model. It suggests that `shot Zones` is the most of the useful feature in location features. 

However, AUC did not improve as much.